# 03 Exploratory Data Analysis


**Sections:**
1. Setup
2. Univariate distributions
3. Categorical breakdowns
4. Bivariate relationships
5. Correlation heatmap
6. Risk-level profiling
7. Academic year trends
8. EDA summary

## 3.1 Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

df.head()

## 3.2 Univariate Distributions — Key Outcome Variables

In [ ]:
outcome_cols = ['burnout_score', 'mental_health_index', 'dropout_risk', 'stress_level']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, col in zip(axes, outcome_cols):
    sns.histplot(data=df, x=col, bins=40, kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col.replace("_", " ").title()}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Outcome Variable Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Predictor distributions
predictor_cols = ['anxiety_score', 'depression_score', 'sleep_hours',
                  'study_hours_per_day', 'physical_activity', 'social_support',
                  'screen_time', 'internet_usage', 'financial_stress',
                  'family_expectation', 'exam_pressure', 'academic_performance']

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for ax, col in zip(axes, predictor_cols):
    sns.histplot(data=df, x=col, bins=40, kde=True, ax=ax, color='teal')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Predictor Variable Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3.3 Categorical Breakdowns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gender distribution
gender_counts = df['gender'].value_counts()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#5B9BD5', '#ED7D31'], startangle=90)
axes[0].set_title('Gender Distribution')

# Risk level distribution
risk_counts = df['risk_level'].value_counts()
axes[1].pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%',
            colors=['#70AD47', '#FF0000'], startangle=90)
axes[1].set_title('Risk Level Distribution')

# Academic year distribution
year_counts = df['academic_year'].value_counts().sort_index()
axes[2].bar(year_counts.index.astype(str), year_counts.values, color='#4472C4')
axes[2].set_title('Students by Academic Year')
axes[2].set_xlabel('Academic Year')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3.4 Bivariate Relationships

In [ ]:
# Burnout score vs key predictors
predictors_vs_burnout = ['stress_level', 'anxiety_score', 'depression_score',
                          'sleep_hours', 'study_hours_per_day', 'social_support']

# Sample 10k rows for scatter plots (performance)
sample = df.sample(n=min(10000, len(df)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, col in zip(axes, predictors_vs_burnout):
    ax.scatter(sample[col], sample['burnout_score'], alpha=0.15, s=5, color='steelblue')
    # Add trend line
    z = np.polyfit(sample[col], sample['burnout_score'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(sample[col].min(), sample[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r-', linewidth=2)
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Burnout Score')
    ax.set_title(f'Burnout vs {col.replace("_", " ").title()}')

plt.suptitle('Burnout Score vs Key Predictors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: burnout and dropout risk by risk level
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=df, x='risk_level', y='burnout_score', palette='Set2', ax=axes[0])
axes[0].set_title('Burnout Score by Risk Level')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Burnout Score')

sns.boxplot(data=df, x='risk_level', y='dropout_risk', palette='Set2', ax=axes[1])
axes[1].set_title('Dropout Risk by Risk Level')
axes[1].set_xlabel('Risk Level')
axes[1].set_ylabel('Dropout Risk')

plt.tight_layout()
plt.show()

In [ ]:
# Burnout score by gender and academic year
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=df, x='gender', y='burnout_score', palette='pastel', ax=axes[0])
axes[0].set_title('Burnout Score by Gender')

sns.boxplot(data=df, x='academic_year', y='burnout_score', palette='pastel', ax=axes[1])
axes[1].set_title('Burnout Score by Academic Year')

plt.tight_layout()
plt.show()

## 3.5 Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Correlation Matrix — All Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with burnout_score
burnout_corr = corr_matrix['burnout_score'].drop('burnout_score').sort_values(key=abs, ascending=False)
print('Top correlations with burnout_score:')
print(burnout_corr.to_string())

## 3.6 Risk-Level Profiling

In [ ]:
risk_profile = df.groupby('risk_level')[num_cols].mean().T
print('Mean values by risk level:')
risk_profile

In [ ]:
# Radar-style bar chart comparing High vs Low risk profiles
key_metrics = ['burnout_score', 'stress_level', 'anxiety_score', 'depression_score',
               'dropout_risk', 'sleep_hours', 'social_support', 'physical_activity',
               'academic_performance']

risk_means = df.groupby('risk_level')[key_metrics].mean()

x = np.arange(len(key_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(16, 6))
for i, (level, row) in enumerate(risk_means.iterrows()):
    ax.bar(x + i * width, row.values, width, label=level)

ax.set_xticks(x + width / 2)
ax.set_xticklabels([m.replace('_', '\n') for m in key_metrics], fontsize=9)
ax.set_title('Mean Metric Values by Risk Level', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Value')
ax.legend(title='Risk Level')
plt.tight_layout()
plt.show()

## 3.7 Academic Year Trends

In [ ]:
year_trends = df.groupby('academic_year')[['burnout_score', 'stress_level',
                                            'anxiety_score', 'dropout_risk',
                                            'academic_performance']].mean()

fig, ax = plt.subplots(figsize=(12, 6))
for col in year_trends.columns:
    ax.plot(year_trends.index, year_trends[col], marker='o', label=col.replace('_', ' ').title())

ax.set_xlabel('Academic Year')
ax.set_ylabel('Mean Score')
ax.set_title('Key Metrics Trend Across Academic Years', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks([1, 2, 3, 4])
plt.tight_layout()
plt.show()

In [ ]:
# High-risk percentage by academic year
high_risk_pct = (
    df.groupby('academic_year')
    .apply(lambda g: (g['risk_level'] == 'High').mean() * 100)
    .reset_index(name='high_risk_pct')
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(high_risk_pct['academic_year'].astype(str), high_risk_pct['high_risk_pct'],
       color=['#4472C4', '#ED7D31', '#A9D18E', '#FF0000'])
ax.set_title('% High-Risk Students by Academic Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Academic Year')
ax.set_ylabel('High Risk %')
for i, v in enumerate(high_risk_pct['high_risk_pct']):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 3.8 Sleep and Screen Time Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sleep hours vs burnout
sleep_burnout = df.groupby(pd.cut(df['sleep_hours'], bins=6))['burnout_score'].mean()
sleep_burnout.plot(kind='bar', ax=axes[0], color='steelblue', rot=30)
axes[0].set_title('Mean Burnout Score by Sleep Hours Bin')
axes[0].set_xlabel('Sleep Hours')
axes[0].set_ylabel('Mean Burnout Score')

# Screen time vs burnout
screen_burnout = df.groupby(pd.cut(df['screen_time'], bins=6))['burnout_score'].mean()
screen_burnout.plot(kind='bar', ax=axes[1], color='coral', rot=30)
axes[1].set_title('Mean Burnout Score by Screen Time Bin')
axes[1].set_xlabel('Screen Time (hours)')
axes[1].set_ylabel('Mean Burnout Score')

plt.tight_layout()
plt.show()

## 3.9 EDA Summary

| Finding | Business Signal |
|---|---|
| Burnout score is right-skewed | Most students have moderate burnout; a tail of severely affected students needs targeted intervention |
| Stress and anxiety are the strongest positive correlates of burnout | Stress-reduction programmes will have the highest ROI |
| Social support and sleep hours are negatively correlated with burnout | Peer support and sleep hygiene campaigns are protective |
| High-risk students show markedly higher burnout, dropout risk, and depression | Risk-level segmentation is a reliable triage tool |
| Burnout tends to increase across academic years | Year 3 and 4 students need proactive outreach |
| Physical activity is negatively correlated with burnout | Wellness programmes targeting sedentary students are warranted |
